In [ ]:
import pandas as pd
from ortools.sat.python import cp_model

In [ ]:
people_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main People')
jobs_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main Desk')

In [ ]:
grade_map = {'A':1, 'B':2, 'C':3, 'D':4, 'F':5}
sec_map = {"CAT1":1, "CAT2":2, "CAT3":3, "CAT4":4, "CAT5":5, "CAT6":6, "CAT7":7, "CAT8":8, "CAT9":9, "CAT10":10, "ANY":0}
edu_map = {'Doc': 1, 'Degree':2, 'Uni':2, 'Dipolma':3, 'O Level':4, 'N Level':5}
health_map = {'Fit':1, 'Semi Fit':2, 'Not Fit': 3}


people_df['Grade'] = people_df['Grade'].replace(grade_map)
people_df['Edu_Type'] = people_df['Edu_Type'].replace(edu_map)
people_df['Health'] = people_df['Health'].replace(health_map)
people_df['Security'] = people_df['Security'].replace(sec_map)

jobs_df['Req_Grade'] = jobs_df['Req_Grade'].replace(grade_map)
jobs_df['Req_Edu'] = jobs_df['Req_Edu'].replace(edu_map)
jobs_df['Req_Health'] = jobs_df['Req_Health'].replace(health_map)
jobs_df['Sec_Clerance'] = jobs_df['Sec_Clerance'].replace(sec_map)


In [ ]:
model = cp_model.CpModel()
solver = cp_model.CpSolver()

no_of_people = len(people_df)
no_of_desk = len(jobs_df)

assignments = {}

In [ ]:
jobs_df.head(1)

In [ ]:
people_df.head(1)

In [ ]:
for i in range(no_of_people):
    for j in range(no_of_desk):
        person = people_df.loc[i]
        job = jobs_df.loc[j]

        health_ok = person["Health"] >= job["Req_Health"]
        sec_ok = True if job["Sec_Clerance"] == 0 else person["Security"] == job["Sec_Clerance"]

        if health_ok and sec_ok:
            assignments[i,j] = model.NewBoolVar(f"x_{i}_{j}")


In [ ]:
for i in range(no_of_people):
    for j in range(no_of_desk):
        if (i,j) in assignments:
            model.Add(
                people_df.loc[i, "Health"] <= jobs_df.loc[j, "Req_Health"]
            ).OnlyEnforceIf(assignments[i,j])

            if jobs_df.loc[j, "Sec_Clerance"] != 0:
                model.Add(
                    people_df.loc[i, "Security"] == jobs_df.loc[j, "Sec_Clerance"]
                ).OnlyEnforceIf(assignments[i,j])

            model.Add(
                people_df.loc[i, "Health"] >= jobs_df.loc[j, "Req_Health"]
            ).OnlyEnforceIf(assignments[i,j])


In [ ]:
for i in range(no_of_people):
    feasible_jobs = [j for j in range(no_of_desk) if (i,j) in assignments]
    if feasible_jobs:
        model.Add(sum(assignments[i,j] for j in feasible_jobs) == 1)
    else:
        print(f"Warning: Person {people_df.loc[i,'Name_ID']} has no feasible job!")


In [ ]:
educational_difference = sum(
    (jobs_df.loc[j, "Req_Edu"] - people_df.loc[i, "Edu_Type"]) * assignments[i,j]
    for i in range(no_of_people)
    for j in range(no_of_desk)
    if (i,j) in assignments and jobs_df.loc[j, "Req_Edu"] > people_df.loc[i, "Edu_Type"]
)

total_edu_penalty = model.NewIntVar(0, 10000, "total_edu_penalty")
model.Add(total_edu_penalty == educational_difference)



In [ ]:
model.Minimize(total_edu_penalty)
status = solver.Solve(model)
print(f"Solver status: {solver.StatusName(status)}")

assigned_rows = []
for (i,j), var in assignments.items():
    if solver.Value(var):
        assigned_rows.append({
            "Person_ID": people_df.loc[i, "Name_ID"],
            "Person_Name": people_df.loc[i, "Name"],
            "Desk_ID": jobs_df.loc[j, "Desk_ID"],
            "Education_Difference": max(0, jobs_df.loc[j, "Req_Edu"] - people_df.loc[i, "Edu_Type"])
        })

assigned_df = pd.DataFrame(assigned_rows)
assigned_df.to_excel("data_set/assignment_results.xlsx", index=False)
print("Assignments saved to assignment_results.xlsx")
